# Phase 3: Full Analysis (5M tokens per corpus, 20M total, two levels)

**Interpretability of Portuguese Language Models via Sparse Autoencoders**

---

**Goal:** Scale the pilot analysis to the full experiment.

**What this notebook does:**
1. Processes **5M tokens per corpus** (4 corpora = 20M tokens total)
2. **Two comparison levels:**
   - Level 1 (Web-crawl): FineWeb-2 PT vs FineWeb EN
   - Level 2 (Wikipedia): Wikipedia PT vs Wikipedia EN
3. Computes **LSI at both levels** for triangulation
4. Selects **150 features** for manual analysis (50 PT + 50 cross + 50 EN)
5. Extracts **top-20 activating examples** per feature
6. Generates full visualizations

**Prerequisites:**
- GPU with ≥16GB VRAM (Colab T4)
- [Accept the Gemma 2 license](https://huggingface.co/google/gemma-2-2b)
- Estimated runtime: ~2-3 hours on T4

In [ ]:
!pip install sae-lens transformer-lens datasets -q
print()
print("Installation complete. Restart the runtime before continuing:")
print("  Runtime → Restart session (ou Ctrl+M .)")
print("  Then skip this cell and run from the next one.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "./checkpoints/"  # adjust to your preferred path (e.g., ./data/checkpoints/ for Colab)
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print('Usando Apple Silicon (MPS)')
else:
    device = 'cpu'
    print('No GPU detected. Processing will be very slow.')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from sae_lens import SAE, HookedSAETransformer

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"

print("Loading Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.float16,
)
print(f"Model: {model.cfg.model_name} | Layers: {model.cfg.n_layers} | d_model: {model.cfg.d_model}")

print()
print(f"Loading SAE: {SAE_ID}...")
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
print(f"SAE: {sae.cfg.d_sae} features | d_in: {sae.cfg.d_in} | hook: {HOOK_NAME}")

---
# Phase 3: Data Collection (4 corpora)

We collect tokens from **4 sources** for two-level comparison:

| Level | PT Corpus | EN Corpus | Purpose |
|---|---|---|---|
| **1 — Web-crawl** | FineWeb-2 PT | FineWeb EN | Realistic comparison (may have domain confounders) |
| **2 — Wikipedia** | Wikipedia PT | Wikipedia EN | Controlled comparison (isolates the language effect) |

**Triangulation:** PT-specific features at **both levels** = strong linguistic evidence.

In [ ]:
N_TOKENS = 5_000_000     # 5M tokens per corpus (adjust if needed)
SEQ_LEN = 256
BATCH_SIZE = 8
MIN_ACTS = 100           # minimum activations to consider a feature active
LSI_THRESHOLD = 0.3      # threshold to classify as language-specific
N_SELECT = 50            # features per category (PT, EN, cross-lingual)
N_EXAMPLES = 20          # activating examples per feature

n_seqs = N_TOKENS // SEQ_LEN
n_batches = n_seqs // BATCH_SIZE

print(f"Tokens per corpus: {N_TOKENS:,}")
print(f"Total tokens (4 corpora): {N_TOKENS * 4:,}")
print(f"Sequences per corpus: {n_seqs:,} ({SEQ_LEN} tokens each)")
print(f"Batches per corpus: {n_batches:,} (batch size {BATCH_SIZE})")
print(f"Selected features: {N_SELECT * 3} (3 × {N_SELECT})")

In [ ]:
from datasets import load_dataset

tokenizer = model.tokenizer

def collect_tokens(dataset, n_tokens, text_field="text", desc="Tokenizando"):
    all_ids = []
    n_articles = 0
    for article in tqdm(dataset, desc=desc):
        text = article[text_field]
        if not text or len(text) < 50:
            continue
        ids = tokenizer.encode(text, add_special_tokens=False)
        all_ids.extend(ids)
        n_articles += 1
        if len(all_ids) >= n_tokens:
            break
    all_ids = all_ids[:n_tokens]
    n_seqs = len(all_ids) // SEQ_LEN
    tokens = torch.tensor(all_ids[:n_seqs * SEQ_LEN], dtype=torch.long).reshape(n_seqs, SEQ_LEN)
    print(f"  {n_articles} texts -> {tokens.numel():,} tokens ({tokens.shape[0]} seqs)")
    return tokens

print("=" * 60)
print("LEVEL 2: WIKIPEDIA (controlled comparison)")
print("=" * 60)

print("\nLoading Wikipedia PT...")
wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train", streaming=True)
tokens_wiki_pt = collect_tokens(wiki_pt, N_TOKENS, desc="Wiki PT")

print("\nLoading Wikipedia EN...")
wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
tokens_wiki_en = collect_tokens(wiki_en, N_TOKENS, desc="Wiki EN")

print()
print("=" * 60)
print("LEVEL 1: WEB-CRAWL (realistic comparison)")
print("=" * 60)

print("\nLoading FineWeb-2 PT...")
web_pt = load_dataset("HuggingFaceFW/fineweb-2", "por_Latn", split="train", streaming=True)
tokens_mc4_pt = collect_tokens(web_pt, N_TOKENS, desc="FineWeb PT")

print("\nLoading FineWeb EN...")
web_en = load_dataset("HuggingFaceFW/fineweb", "sample-10BT", split="train", streaming=True)
tokens_c4_en = collect_tokens(web_en, N_TOKENS, desc="FineWeb EN")

print()
print("Collection done!")
print(f"Total: {(tokens_wiki_pt.numel() + tokens_wiki_en.numel() + tokens_mc4_pt.numel() + tokens_c4_en.numel()):,} tokens")

---
# Phase 3: Activation Processing

We pass all tokens through **Gemma 2 2B** and extract activations at **layer 13**, encoding with the **16k-feature SAE**.

Estimated runtime: ~30 min per corpus on T4 (~2h total).

In [ ]:
@torch.no_grad()
def compute_feature_stats(model, sae, tokens, batch_size=BATCH_SIZE, desc=""):
    n_features = sae.cfg.d_sae
    hook = HOOK_NAME
    counts = torch.zeros(n_features, device=device)
    sums = torch.zeros(n_features, device=device)
    maxvals = torch.zeros(n_features, device=device)
    total = 0

    n_batches = (len(tokens) + batch_size - 1) // batch_size
    t0 = time.time()
    for i in tqdm(range(0, len(tokens), batch_size), desc=desc, total=n_batches):
        batch = tokens[i:i+batch_size].to(device)
        _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook)
        acts = cache[hook]
        feat_acts = sae.encode(acts)

        active = feat_acts > 0
        counts += active.float().sum(dim=(0, 1))
        sums += feat_acts.sum(dim=(0, 1))
        maxvals = torch.max(maxvals, feat_acts.amax(dim=(0, 1)))
        total += batch.numel()

        del cache, acts, feat_acts, active
        if device == "cuda":
            torch.cuda.empty_cache()

    elapsed = time.time() - t0
    print(f"  {total:,} tokens | {(counts > 0).sum().item():,} active features | {elapsed:.0f}s")
    return {"counts": counts.cpu(), "sums": sums.cpu(), "max": maxvals.cpu(), "total_tokens": total}

In [ ]:
import os

# --- Wiki PT ---
if os.path.exists("stats_wiki_pt.pt"):
    print("Loading checkpoint stats_wiki_pt.pt...")
    stats_wiki_pt = torch.load("stats_wiki_pt.pt")
else:
    print("Wikipedia PT...")
    stats_wiki_pt = compute_feature_stats(model, sae, tokens_wiki_pt, desc="Wiki PT")
    torch.save(stats_wiki_pt, SAVE_DIR + "stats_wiki_pt.pt")
    print("Checkpoint saved: stats_wiki_pt.pt")

# --- Wiki EN ---
if os.path.exists("stats_wiki_en.pt"):
    print("Loading checkpoint stats_wiki_en.pt...")
    stats_wiki_en = torch.load("stats_wiki_en.pt")
else:
    print("\nWikipedia EN...")
    stats_wiki_en = compute_feature_stats(model, sae, tokens_wiki_en, desc="Wiki EN")
    torch.save(stats_wiki_en, SAVE_DIR + "stats_wiki_en.pt")
    print("Checkpoint saved: stats_wiki_en.pt")

In [ ]:
import os

# --- FineWeb PT ---
if os.path.exists("stats_mc4_pt.pt"):
    print("Loading checkpoint stats_mc4_pt.pt...")
    stats_mc4_pt = torch.load("stats_mc4_pt.pt")
else:
    print("FineWeb PT...")
    stats_mc4_pt = compute_feature_stats(model, sae, tokens_mc4_pt, desc="FineWeb PT")
    torch.save(stats_mc4_pt, SAVE_DIR + "stats_mc4_pt.pt")
    print("Checkpoint saved: stats_mc4_pt.pt")

# --- FineWeb EN ---
if os.path.exists("stats_c4_en.pt"):
    print("Loading checkpoint stats_c4_en.pt...")
    stats_c4_en = torch.load("stats_c4_en.pt")
else:
    print("\nFineWeb EN...")
    stats_c4_en = compute_feature_stats(model, sae, tokens_c4_en, desc="FineWeb EN")
    torch.save(stats_c4_en, SAVE_DIR + "stats_c4_en.pt")
    print("Checkpoint saved: stats_c4_en.pt")

---
# Phase 3: Analysis — Two-level LSI + Feature Selection

We compute the LSI separately for each level and use **triangulation** to distinguish linguistic effects from domain effects.

In [ ]:
# --- LSI Level 2: Wikipedia ---
freq_wiki_pt = stats_wiki_pt["counts"] / stats_wiki_pt["total_tokens"]
freq_wiki_en = stats_wiki_en["counts"] / stats_wiki_en["total_tokens"]
lsi_wiki = (freq_wiki_pt - freq_wiki_en) / (freq_wiki_pt + freq_wiki_en + 1e-10)

# --- LSI Level 1: Web-crawl ---
freq_mc4_pt = stats_mc4_pt["counts"] / stats_mc4_pt["total_tokens"]
freq_c4_en = stats_c4_en["counts"] / stats_c4_en["total_tokens"]
lsi_web = (freq_mc4_pt - freq_c4_en) / (freq_mc4_pt + freq_c4_en + 1e-10)

# --- Active features ---
total_counts = (stats_wiki_pt["counts"] + stats_wiki_en["counts"]
                + stats_mc4_pt["counts"] + stats_c4_en["counts"])
alive = total_counts > 0
active = total_counts >= MIN_ACTS

n_total = sae.cfg.d_sae
n_dead = (~alive).sum().item()
n_alive = alive.sum().item()
n_active = active.sum().item()

# --- Triangulation ---
pt_wiki = (lsi_wiki > LSI_THRESHOLD) & active
pt_web = (lsi_web > LSI_THRESHOLD) & active
pt_both = pt_wiki & pt_web   # strong linguistic evidence
pt_wiki_only = pt_wiki & ~pt_web
pt_web_only = pt_web & ~pt_wiki

en_wiki = (lsi_wiki < -LSI_THRESHOLD) & active
en_web = (lsi_web < -LSI_THRESHOLD) & active
en_both = en_wiki & en_web

cross_wiki = (lsi_wiki.abs() <= LSI_THRESHOLD) & active
cross_web = (lsi_web.abs() <= LSI_THRESHOLD) & active
cross_both = cross_wiki & cross_web

print("=" * 65)
print("GENERAL STATISTICS")
print("=" * 65)
print(f"Total features:              {n_total:,}")
print(f"Dead features:             {n_dead:,} ({n_dead/n_total*100:.1f}%)")
print(f"Alive features:              {n_alive:,} ({n_alive/n_total*100:.1f}%)")
print(f"Active features (>={MIN_ACTS}):     {n_active:,}")

print()
print("=" * 65)
print("LSI — LEVEL 2 (WIKIPEDIA)")
print("=" * 65)
lsi_w = lsi_wiki[active]
print(f"  Mean:     {lsi_w.mean().item():.4f}")
print(f"  Mediana:  {lsi_w.median().item():.4f}")
print(f"  PT-predominantes (>{LSI_THRESHOLD}):  {pt_wiki.sum().item()}")
print(f"  EN-predominantes (<{-LSI_THRESHOLD}): {en_wiki.sum().item()}")
print(f"  Cross-lingual:              {cross_wiki.sum().item()}")

print()
print("=" * 65)
print("LSI — LEVEL 1 (WEB-CRAWL)")
print("=" * 65)
lsi_wc = lsi_web[active]
print(f"  Mean:     {lsi_wc.mean().item():.4f}")
print(f"  Mediana:  {lsi_wc.median().item():.4f}")
print(f"  PT-predominantes (>{LSI_THRESHOLD}):  {pt_web.sum().item()}")
print(f"  EN-predominantes (<{-LSI_THRESHOLD}): {en_web.sum().item()}")
print(f"  Cross-lingual:              {cross_web.sum().item()}")

print()
print("=" * 65)
print("TRIANGULATION")
print("=" * 65)
print(f"  PT at BOTH levels:        {pt_both.sum().item()}  ← strong linguistic evidence")
print(f"  PT Wikipedia only:        {pt_wiki_only.sum().item()}  ← possible style effect")
print(f"  PT Web-crawl only:        {pt_web_only.sum().item()}  ← possible domain effect")
print(f"  EN at BOTH levels:        {en_both.sum().item()}")
print(f"  Cross-lingual em ambos:    {cross_both.sum().item()}")

In [ ]:
# --- Combined LSI (average of the two levels) ---
lsi_combined = (lsi_wiki + lsi_web) / 2

# --- Selection: Top-50 PT ---
lsi_pt_rank = lsi_combined.clone()
lsi_pt_rank[~active] = -2
top_pt_vals, top_pt_idx = lsi_pt_rank.topk(N_SELECT)

# --- Selection: Top-50 EN ---
lsi_en_rank = lsi_combined.clone()
lsi_en_rank[~active] = 2
top_en_vals, top_en_idx = lsi_en_rank.topk(N_SELECT, largest=False)

# --- Selection: Top-50 Cross-lingual (|LSI| closest to 0, with high frequency) ---
total_freq = freq_wiki_pt + freq_wiki_en + freq_mc4_pt + freq_c4_en
cross_score = lsi_combined.abs().clone()
cross_score[~active] = 999
min_cross_freq = total_freq[active].median()
cross_score[total_freq < min_cross_freq] = 999
top_cross_vals, top_cross_idx = cross_score.topk(N_SELECT, largest=False)

# --- Consolidar ---
selected_pt = top_pt_idx.tolist()
selected_en = top_en_idx.tolist()
selected_cross = top_cross_idx.tolist()
all_selected = selected_pt + selected_cross + selected_en

print("=" * 90)
print(f"SELECTED FEATURES: {len(all_selected)} total")
print("=" * 90)

def print_table(label, indices, lsi_tensor):
    fmt = '{:<4} {:<8} {:<10} {:<10} {:<10} {:<10} {:<10}'
    print(f"\n{'—' * 90}")
    print(f"{label} ({len(indices)} features)")
    print('—' * 90)
    print(fmt.format('#', 'Feature', 'LSI Wiki', 'LSI Web', 'LSI Comb', 'Freq PT*', 'Freq EN*'))
    print('-' * 90)
    for i, idx in enumerate(indices[:20]):
        freq_pt_total = (freq_wiki_pt[idx] + freq_mc4_pt[idx]).item() / 2
        freq_en_total = (freq_wiki_en[idx] + freq_c4_en[idx]).item() / 2
        print(fmt.format(
            i+1, idx,
            f'{lsi_wiki[idx].item():+.4f}',
            f'{lsi_web[idx].item():+.4f}',
            f'{lsi_combined[idx].item():+.4f}',
            f'{freq_pt_total:.6f}',
            f'{freq_en_total:.6f}',
        ))
    if len(indices) > 20:
        print(f"  ... (+{len(indices) - 20} features not shown)")
    print("  * Freq = average of the two corpora per language")

print_table("TOP PT-SPECIFIC", selected_pt, lsi_combined)
print_table("TOP CROSS-LINGUAL", selected_cross, lsi_combined)
print_table("TOP EN-SPECIFIC (control)", selected_en, lsi_combined)

# Mark which are consistent across both levels
n_pt_consistent = sum(1 for idx in selected_pt if pt_both[idx].item())
print(f"\n>>> Of the {N_SELECT} selected PT features, {n_pt_consistent} are consistent at both levels.")

In [ ]:
@torch.no_grad()
def find_top_examples(model, sae, tokens, feature_indices,
                      n_examples=N_EXAMPLES, batch_size=BATCH_SIZE, max_batches=500):
    hook = HOOK_NAME
    feat_list = list(feature_indices)
    results = {f: [] for f in feat_list}
    tok = model.tokenizer

    actual_batches = min(max_batches, (len(tokens) + batch_size - 1) // batch_size)
    for i in tqdm(range(0, actual_batches * batch_size, batch_size),
                  desc="Searching for examples", total=actual_batches):
        if i >= len(tokens):
            break
        batch = tokens[i:i+batch_size].to(device)
        _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook)
        acts = cache[hook]
        feat_acts = sae.encode(acts)

        for f_idx in feat_list:
            vals = feat_acts[:, :, f_idx]
            for b in range(batch.shape[0]):
                max_val, max_pos = vals[b].max(dim=0)
                if max_val.item() > 0:
                    pos = max_pos.item()
                    start = max(0, pos - 10)
                    end = min(batch.shape[1], pos + 15)
                    ctx = tok.decode(batch[b, start:end].tolist())
                    results[f_idx].append({"act": max_val.item(), "context": ctx})

        del cache, acts, feat_acts
        if device == "cuda":
            torch.cuda.empty_cache()

    for f in results:
        results[f].sort(key=lambda x: x["act"], reverse=True)
        results[f] = results[f][:n_examples]
    return results

# Find PT examples (in Wiki PT, cleaner corpus)
print("Finding examples for PT-specific features (Wiki PT)...")
examples_pt = find_top_examples(model, sae, tokens_wiki_pt, selected_pt)

# Search cross-lingual examples (in Wiki PT)
print("\nFinding examples for cross-lingual features (Wiki PT)...")
examples_cross = find_top_examples(model, sae, tokens_wiki_pt, selected_cross)

# Search EN examples (in Wiki EN)
print("\nFinding examples for EN-specific features (Wiki EN)...")
examples_en = find_top_examples(model, sae, tokens_wiki_en, selected_en)

In [ ]:
print("=" * 80)
print("ACTIVATING EXAMPLES — TOP-20 PT-SPECIFIC FEATURES")
print("=" * 80)

for rank, f_idx in enumerate(selected_pt[:20]):
    print()
    print("-" * 70)
    consistent = "AMBOS NÍVEIS" if pt_both[f_idx].item() else "apenas 1 nível"
    print(f'Feature {f_idx} | LSI wiki={lsi_wiki[f_idx].item():+.4f} | '
          f'LSI web={lsi_web[f_idx].item():+.4f} | Rank #{rank+1} | {consistent}')
    print("-" * 70)
    exs = examples_pt.get(f_idx, [])
    if not exs:
        print("  (no examples)")
        continue
    for ex in exs[:5]:
        print(f'  [{ex["act"]:.2f}] ...{ex["context"]}...')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (A) Histograma LSI Wikipedia
lsi_w_np = lsi_wiki[active].numpy()
axes[0, 0].hist(lsi_w_np, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].axvline(LSI_THRESHOLD, color='red', linestyle=':', linewidth=1)
axes[0, 0].axvline(-LSI_THRESHOLD, color='red', linestyle=':', linewidth=1)
axes[0, 0].axvline(0, color='black', linestyle='--', linewidth=0.8)
axes[0, 0].set_xlabel('LSI')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('(A) LSI Distribution — Wikipedia')

# (B) Histograma LSI Web-crawl
lsi_wc_np = lsi_web[active].numpy()
axes[0, 1].hist(lsi_wc_np, bins=100, color='darkorange', edgecolor='white', alpha=0.8)
axes[0, 1].axvline(LSI_THRESHOLD, color='red', linestyle=':', linewidth=1)
axes[0, 1].axvline(-LSI_THRESHOLD, color='red', linestyle=':', linewidth=1)
axes[0, 1].axvline(0, color='black', linestyle='--', linewidth=0.8)
axes[0, 1].set_xlabel('LSI')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('(B) LSI Distribution — Web-crawl')

# (C) Triangulation: LSI Wiki vs LSI Web
lsi_w_a = lsi_wiki[active].numpy()
lsi_wc_a = lsi_web[active].numpy()
colors = np.where(
    (lsi_w_a > LSI_THRESHOLD) & (lsi_wc_a > LSI_THRESHOLD), 'red',
    np.where(
        (lsi_w_a < -LSI_THRESHOLD) & (lsi_wc_a < -LSI_THRESHOLD), 'blue',
        'gray'
    )
)
axes[1, 0].scatter(lsi_w_a, lsi_wc_a, c=colors, s=3, alpha=0.4)
axes[1, 0].axhline(LSI_THRESHOLD, color='red', linestyle=':', alpha=0.5)
axes[1, 0].axhline(-LSI_THRESHOLD, color='red', linestyle=':', alpha=0.5)
axes[1, 0].axvline(LSI_THRESHOLD, color='red', linestyle=':', alpha=0.5)
axes[1, 0].axvline(-LSI_THRESHOLD, color='red', linestyle=':', alpha=0.5)
axes[1, 0].plot([-1, 1], [-1, 1], 'k--', linewidth=0.8, alpha=0.4)
axes[1, 0].set_xlabel('LSI (Wikipedia)')
axes[1, 0].set_ylabel('LSI (Web-crawl)')
axes[1, 0].set_title('(C) Triangulation: Wiki vs Web LSI')
axes[1, 0].set_xlim(-1.05, 1.05)
axes[1, 0].set_ylim(-1.05, 1.05)

# (D) Heatmap: Top-20 PT features, 4 corpora
top20_pt = selected_pt[:20]
data = np.column_stack([
    freq_wiki_pt[top20_pt].numpy(),
    freq_wiki_en[top20_pt].numpy(),
    freq_mc4_pt[top20_pt].numpy(),
    freq_c4_en[top20_pt].numpy(),
])
im = axes[1, 1].imshow(data, aspect='auto', cmap='YlOrRd')
axes[1, 1].set_xticks([0, 1, 2, 3])
axes[1, 1].set_xticklabels(['Wiki PT', 'Wiki EN', 'FineWeb PT', 'FineWeb EN'], fontsize=8)
axes[1, 1].set_yticks(range(len(top20_pt)))
labels = [f'F{idx}' for idx in top20_pt]
axes[1, 1].set_yticklabels(labels, fontsize=7)
axes[1, 1].set_title('(D) Top-20 PT Features: Freq by Corpus')
plt.colorbar(im, ax=axes[1, 1], label='Activation freq', shrink=0.8)

plt.tight_layout()
plt.savefig('fig_phase3_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_phase3_overview.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 16))

data50 = np.column_stack([
    freq_wiki_pt[selected_pt].numpy(),
    freq_wiki_en[selected_pt].numpy(),
    freq_mc4_pt[selected_pt].numpy(),
    freq_c4_en[selected_pt].numpy(),
])
im = ax.imshow(data50, aspect='auto', cmap='YlOrRd')
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['Wiki PT', 'Wiki EN', 'FineWeb PT', 'FineWeb EN'])
ax.set_yticks(range(len(selected_pt)))
ylabels = []
for idx in selected_pt:
    marker = "★" if pt_both[idx].item() else " "
    ylabels.append(f'{marker} F{idx} (LSI={lsi_combined[idx].item():+.2f})')
ax.set_yticklabels(ylabels, fontsize=6)
ax.set_title(f'Top-{N_SELECT} PT-Specific Features — Activation Frequency\n(★ = consistent across both levels)')
plt.colorbar(im, label='Activation frequency', shrink=0.6)
plt.tight_layout()
plt.savefig('fig_phase3_heatmap_top50_pt.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_phase3_heatmap_top50_pt.png")

In [ ]:
results = {
    "config": {
        "n_tokens": N_TOKENS,
        "seq_len": SEQ_LEN,
        "layer": LAYER,
        "min_acts": MIN_ACTS,
        "lsi_threshold": LSI_THRESHOLD,
        "n_select": N_SELECT,
    },
    "stats": {
        "wiki_pt": stats_wiki_pt,
        "wiki_en": stats_wiki_en,
        "mc4_pt": stats_mc4_pt,
        "c4_en": stats_c4_en,
    },
    "frequencies": {
        "wiki_pt": freq_wiki_pt,
        "wiki_en": freq_wiki_en,
        "mc4_pt": freq_mc4_pt,
        "c4_en": freq_c4_en,
    },
    "lsi": {
        "wiki": lsi_wiki,
        "web": lsi_web,
        "combined": lsi_combined,
    },
    "masks": {
        "active": active,
        "pt_both": pt_both,
        "pt_wiki_only": pt_wiki_only,
        "pt_web_only": pt_web_only,
    },
    "selected": {
        "pt": selected_pt,
        "cross": selected_cross,
        "en": selected_en,
        "all": all_selected,
    },
    "examples": {
        "pt": examples_pt,
        "cross": examples_cross,
        "en": examples_en,
    },
}

torch.save(results, "phase3_results.pt")
print("Full results saved to phase3_results.pt")
print()
print("Generated files:")
print("  phase3_results.pt              — all data")
print("  stats_wiki_pt.pt              — checkpoint Wikipedia PT")
print("  stats_wiki_en.pt              — checkpoint Wikipedia EN")
print("  stats_mc4_pt.pt               — checkpoint Web-crawl PT")
print("  stats_c4_en.pt                — checkpoint Web-crawl EN")
print("  fig_phase3_overview.png        — 4 panels (distributions + triangulation + heatmap)")
print("  fig_phase3_heatmap_top50_pt.png — heatmap detailed top-50 PT")